Запуск оценки качества RAG через ragas.

**FAISS ретривер**
- Векторная база лежит в `vector_db/` рядом с проектом.
- Нужны файлы: `index.faiss` и `docs.json` или `docs.jsonl` (поле `text`).
- Пути к локальным моделям задаются через `E5_MODEL_PATH` и `BGE3_MODEL_PATH`.

In [ ]:
from __future__ import annotations

from pathlib import Path
import os

from langchain_gigachat.chat_models import GigaChat
from ragas.llms import LangchainLLMWrapper

from rag_eval.run_eval import run_eval, SentenceTransformer

# ==== НАСТРОЙКИ ====
DATA_DIR = Path('data')
REPORTS_DIR = Path('reports')
VECTOR_DB_DIR = Path('vector_db')
REGISTRY_MODULES = ['rag_systems']
SYSTEM_NAMES = ['e5-large', 'bge-3']

# Пути к локальным моделям (поставь свои пути)
E5_MODEL_PATH = '/models/e5-large'
BGE3_MODEL_PATH = '/models/bge-3'
RETRIEVER_TOP_K = 5

# Пробрасываем окружение для rag_systems.py
os.environ['VECTOR_DB_DIR'] = str(VECTOR_DB_DIR)
os.environ['E5_MODEL_PATH'] = E5_MODEL_PATH
os.environ['BGE3_MODEL_PATH'] = BGE3_MODEL_PATH
os.environ['RETRIEVER_TOP_K'] = str(RETRIEVER_TOP_K)

# Конфиг ретривера/систем для отчёта (попадает в Config)
RUN_CONFIG = {
    'retriever_model': 'e5-large|bge-3',
    'retriever_top_k': RETRIEVER_TOP_K,
    'retriever_db_path': str(VECTOR_DB_DIR),
}

# Локальная модель эмбеддингов для автономной метрики answer_cosine_emb
# Пример: LOCAL_EMB_MODEL = '/models/sentence-transformers/all-MiniLM-L6-v2'
LOCAL_EMB_MODEL = os.environ.get('LOCAL_EMB_MODEL')

# ==== ХЕЛПЕРЫ ====
def latest_xlsx(data_dir: Path) -> Path:
    xlsx_files = sorted(data_dir.glob('*.xlsx'), key=lambda p: p.stat().st_mtime, reverse=True)
    if not xlsx_files:
        raise FileNotFoundError(f'Нет .xlsx в папке {data_dir.resolve()}')
    return xlsx_files[0]

# ==== JUDGE LLM ====
judge = GigaChat()
judge_llm = LangchainLLMWrapper(judge)

# ==== ЛОКАЛЬНЫЕ ЭМБЕДДИНГИ (опционально) ====
local_embedder = None
if LOCAL_EMB_MODEL and SentenceTransformer is not None:
    local_embedder = SentenceTransformer(LOCAL_EMB_MODEL)

# ==== ВХОДНЫЕ ДАННЫЕ ====
GOLD_XLSX = latest_xlsx(DATA_DIR)


In [ ]:
run_eval(
    gold_path=str(GOLD_XLSX),
    registry_modules=REGISTRY_MODULES,
    system_names=SYSTEM_NAMES,
    output_root=str(REPORTS_DIR),
    judge_llm=judge_llm,
    local_embedder=local_embedder,
    run_config=RUN_CONFIG,
)

print('Saved:', REPORTS_DIR.resolve())